# CityBike · Modelleren en bevindingen

Dit notebook bevat de hoofdstukken 3 en 4 van Mehmet Kilinc. Plaats de modelcellen en bevindingen na hoofdstuk 2 in het gezamenlijke notebook. Voer de eerdere hoofdstukken in dezelfde kernel uit.

- EDA en data opschonen leveren `train` en `test`. Gebruik de gedeelde `DataLoader` met het pad naar de map `data/` vanuit de werkmap van de notebookkernel.
- Feature engineering levert `train_fe`, `test_fe`, `CV`, `N_AUTUMN`, `HOLDOUT_START`, `RANDOM_STATE`, `regression_metrics`, `linear_model` en de featurelijsten `BASELINE_FEATURES`, `LINEAR_CAT`, `LINEAR_NUM`, `TREE_FEATURES`, `TS_FEATURES` en `TREND_FEATURES`.
- De extra modelimports hieronder kunnen bij de gezamenlijke imports. De instellingen `MIN_KAGGLE_RMSE` en de map `submissions/` moeten aanwezig zijn voordat hoofdstuk 3 draait. In de eindversie hoort de werkmap de hoofdmap van het project te zijn.
- Neem de bronregels aan het eind mee naar de gezamenlijke bronnenlijst. De tabellen in §4.3 bevatten de eerder genoteerde Kaggle-scores en de open plekken voor nieuwe inzendingen.


In [ ]:
# Bron: Week 10 – Regression Time Series 2 (imports)
# Bron: Week 11 – Time series analysis 3 (SARIMAX)
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

MIN_KAGGLE_RMSE = 80
os.makedirs('submissions', exist_ok=True)


# 3. Opdracht 3: Modelleren

## 3.1 Aanpak en klasse `RegressionModels`

We gebruiken een klasse zoals `RegressionModels` uit de notebooks *Regression Time Series 2*
(opdracht 2, 4, 10, 12 en 13) en *Regression Time Series 3* (De Haagse Hogeschool, z.d.-d,
z.d.-e):

- **constructor**: splitst de data in een trainset en een testset (de laatste 19 dagen, zonder
  husselen) en bewaart de Kaggle-testdata;
- **`tune_params`**: loopt over een dictionary met modellen, parameter-grids en features en zoekt per
  model de beste hyperparameters met `GridSearchCV` en onze tijdreeks-folds (2.1);
- **`predict`**: test het beste model (met de beste parameters, getraind op de hele trainset) op de
  testset met $R^2$, MAE, MSE en RMSE;
- **`show_feature_importances`**: toont de feature importances, als het model die heeft;
- **`make_submission`** (nieuw in de klasse): traint het beste model opnieuw op **alle** data
  (Radhakrishnan, 2023a) en maakt een submission voor Kaggle, zoals in opdracht 3 van
  *Regression 1* (De Haagse Hogeschool, z.d.-b).

Ook de tijdreeksmodellen (SARIMAX, Prophet) en het hybride model schrijven we als
scikit-learn-model (met `fit` en `predict`), zodat ze op precies dezelfde manier getuned en getest
worden. De methode `results_table` en de functie `show_grid` (overzicht van de grid search) komen
niet in de lesstof voor (Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook).


[ChatGPT, 2026. Prompt 10: Resultatentabel en grid search](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)

Prompts 10 t/m 16 zijn gebruikt om de bestaande code in hoofdstuk 3 en 4 beter te begrijpen.

In [ ]:
# Bron: Week 12 – Regression Time Series 3 (klasse RegressionModels)
# Bron: Week 10 – Regression Time Series 2, opdracht 2, 4, 10, 12 en 13
# Bron: Week 8 – Regression 1, opdracht 3 (submission voor Kaggle)
# Prompt 1: Portfolio 2 CityBike notebook (results_table, show_grid)
# Prompt 10: Resultatentabel en grid search

class RegressionModels:
    """Tunen, testen en voorspellen van regressiemodellen.

    Gebaseerd op de klasse uit de lessen Regression Time Series 2 en 3.
    """

    def __init__(self, data, kaggle_test, target, holdout_start, cv):
        self.data = data
        self.target = target
        is_train = data['date_hour'] < holdout_start
        self.train = data[is_train].reset_index(drop=True)    # trainset
        self.test = data[~is_train].reset_index(drop=True)    # testset
        self.kaggle_test = kaggle_test
        self.cv = cv
        self.models = {}
        self.results = {}
        self.test_predictions = {}

    def tune_params(self, models_dict, n_jobs=-1):
        """Zoek per model de beste hyperparameters met GridSearchCV."""
        for name, (model, param_grid, features) in models_dict.items():
            search = GridSearchCV(model, param_grid, cv=self.cv,
                                  scoring='neg_root_mean_squared_error',
                                  n_jobs=n_jobs)
            search.fit(self.train[features], self.train[self.target])
            best = search.best_index_
            folds = np.array(
                [-search.cv_results_[f'split{i}_test_score'][best]
                 for i in range(len(self.cv))])
            self.models[name] = (search.best_estimator_, features)
            self.results[name] = {
                'CV-RMSE': folds.mean(),
                'herfst': folds[:N_AUTUMN].mean(),
                'winter': folds[N_AUTUMN:].mean(),
                'beste parameters': search.best_params_}
            print(f'{name}: CV-RMSE = {folds.mean():.1f} '
                  f'(herfst {folds[:N_AUTUMN].mean():.1f}, '
                  f'winter {folds[N_AUTUMN:].mean():.1f})')
            print(f'  beste parameters: {search.best_params_}')
            self.predict(name)
        return search

    def predict(self, name):
        """Test het beste model op de testset (R2, MAE, MSE en RMSE)."""
        model, features = self.models[name]
        y_pred = np.clip(model.predict(self.test[features]), 0, None)
        self.test_predictions[name] = y_pred
        metrics = regression_metrics(self.test[self.target], y_pred)
        for key, value in metrics.items():
            self.results[name][f'test {key}'] = value
        print(f"  testset: R2 = {metrics['R2']:.3f}, "
              f"MAE = {metrics['MAE']:.1f}, RMSE = {metrics['RMSE']:.1f}")
        return y_pred

    def make_submission(self, name, file_name):
        """Train het beste model opnieuw op alle data en voorspel Kaggle."""
        model, features = self.models[name]
        final_model = clone(model).fit(self.data[features],
                                       self.data[self.target])
        prediction = final_model.predict(self.kaggle_test[features])
        submission = pd.DataFrame({
            'date_hour': self.kaggle_test['date_hour'].dt.strftime(
                '%Y-%m-%d %H:%M:%S'),
            'cnt': np.clip(prediction, 0, None)})
        submission.to_csv(os.path.join('submissions', file_name),
                          index=False)
        self.results[name]['submission'] = file_name
        return final_model

    def show_feature_importances(self, name):
        """Plot de feature importances (alleen als het model die heeft)."""
        model, features = self.models[name]
        if not hasattr(model, 'feature_importances_'):
            print(f'{name} heeft geen feature_importances_.')
            return
        importances = pd.Series(model.feature_importances_, index=features)
        importances.sort_values().plot(
            kind='barh', figsize=(8, 4),
            title=f'Feature importances: {name}')
        plt.show()

    def results_table(self):
        """Alle resultaten, gesorteerd van beste naar slechtste CV-RMSE."""
        table = pd.DataFrame(self.results).T
        baseline_rmse = table.loc['Baseline', 'test RMSE']
        table['verbetering t.o.v. baseline'] = (
            1 - table['test RMSE'].astype(float) / baseline_rmse)
        return table.sort_values('CV-RMSE')


def show_grid(search, top=10):
    """Toon de scores van de geteste hyperparameter-combinaties."""
    table = pd.DataFrame(search.cv_results_)
    # korte namen: 'regressor__model__alpha' -> 'alpha'
    table['parameters'] = table['params'].apply(
        lambda params: ', '.join(f'{key.split("__")[-1]}={value}'
                                 for key, value in params.items()))
    table['CV-RMSE'] = -table['mean_test_score']
    return (table[['parameters', 'CV-RMSE']].sort_values('CV-RMSE')
            .head(top).round(1))


models = RegressionModels(train_fe, test_fe, 'cnt', HOLDOUT_START, CV)
print('trainset:', models.train.shape, '| testset:', models.test.shape,
      '| Kaggle-test:', models.kaggle_test.shape)

## 3.2 Baseline

Een model is pas nuttig als het beter is dan een eenvoudige **baseline** (Radhakrishnan, 2025). Onze
baseline is wat een planner zonder machine learning zou doen: voorspel voor elk uur de **gemiddelde
verhuur op hetzelfde uur en hetzelfde type dag** (werkdag of weekend) in de laatste weken. Het aantal
weken is de enige 'hyperparameter'. De klasse `SeasonalBaseline` komt niet in de lesstof voor
(Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook).


[ChatGPT, 2026. Prompt 11: Seizoensbaseline](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)

In [ ]:
# Bron: Week 8 – slides Veelvoorkomende fouten (wat is je baseline?)
# Prompt 1: Portfolio 2 CityBike notebook (klasse SeasonalBaseline)
# Prompt 11: Seizoensbaseline

class SeasonalBaseline(BaseEstimator, RegressorMixin):
    """Gemiddelde per uur en dagtype over de laatste n_days van de training."""

    def __init__(self, n_days=28):
        self.n_days = n_days

    def fit(self, X, y):
        data = X[['date_hour', 'hour', 'workingday']].copy()
        data['cnt'] = np.asarray(y)
        start = data['date_hour'].max() - pd.Timedelta(days=self.n_days)
        recent = data[data['date_hour'] > start]
        self.profile_ = recent.groupby(['hour', 'workingday'])['cnt'].mean()
        self.overall_ = recent['cnt'].mean()
        return self

    def predict(self, X):
        keys = pd.MultiIndex.from_arrays([X['hour'], X['workingday']])
        return self.profile_.reindex(keys).fillna(self.overall_).to_numpy()


models.tune_params({'Baseline': (SeasonalBaseline(), {'n_days': [14, 28]},
                                 BASELINE_FEATURES)})
baseline_final = models.make_submission('Baseline',
                                        'submission_baseline.csv')

## 3.3 Model 1: Lineaire regressie met regularisatie

**Werking.** Lineaire regressie voorspelt de target als gewogen som van de features:
$$\hat{y}_i = \beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip} = \mathbf{x}_i^\top\boldsymbol{\beta}$$

**Loss function.** De gewichten $\boldsymbol\beta$ worden zo gekozen dat de *loss function* minimaal
is: de gemiddelde gekwadrateerde fout (MSE)
$$J(\boldsymbol\beta) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$
Voor gewone lineaire regressie (OLS) is er een exacte oplossing,
$\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf{y}$; het minimum kun je ook met gradient
descent vinden.

**Regularisatie.** Met veel features (48 dummies voor uur × werkdag, Fourier-termen, weer) en sterk
gecorreleerde features (`temp` en `atemp`, zie 1.8) kunnen de gewichten groot en instabiel worden →
overfitting. Regularisatie voegt een **strafterm** voor grote gewichten toe (James et al., 2021):

- **Ridge (L2):** $J(\boldsymbol\beta) = \text{MSE} + \alpha\sum_{j=1}^{p}\beta_j^2$ → alle gewichten
  worden kleiner (Hoerl & Kennard, 1970);
- **Lasso (L1):** $J(\boldsymbol\beta) = \text{MSE} + \alpha\sum_{j=1}^{p}|\beta_j|$ → sommige
  gewichten worden precies 0, dus ook featureselectie (Tibshirani, 1996).

$\alpha$ is een hyperparameter: een grotere $\alpha$ geeft een eenvoudiger model (meer bias, minder
variantie). Omdat de straf van de grootte van de gewichten afhangt, standaardiseren we de
**numerieke** features. Categorieën worden dummies en worden niet gestandaardiseerd
(Radhakrishnan, 2025).

**Keuzes (zie 2.8).**

- Dagpatroon: dummies voor uur × werkdag (`hour_wd`), plus dummies voor weekdag en weersituatie. De
  eerste categorie laten we weg (`drop='first'`, de dummy trap).
- Trend: `trend` uit `DeterministicProcess` (order 1); jaarseizoen: Fourier-termen (beste orde uit
  2.8).
- **Target $\log(1 + \text{cnt})$.** De effecten zijn multiplicatief (regen kost een percentage van de
  verhuur). Na de log-transformatie worden ze optellend, wat bij een lineair model past.
  `TransformedTargetRegressor` zet de voorspelling terug met $e^{\hat{y}} - 1$.
- Grid: gewone `LinearRegression`, `Ridge` en `Lasso` met verschillende $\alpha$. Net als in
  opdracht 1 van *Regression 1* (De Haagse Hogeschool, z.d.-b) zoeken we de beste parameters met
  grid search en cross-validatie, met een dictionary van modellen en grids zoals de `models_dict`
  uit opdracht 12 van *Regression Time Series 2* (De Haagse Hogeschool, z.d.-d).

In [ ]:
# Bron: Week 8 – Regression 1, opdracht 1 (grid search en CV)
# Bron: Week 10 – Regression Time Series 2, opdracht 12 (models_dict)

linear_grid = [
    {'regressor__model': [LinearRegression()]},
    {'regressor__model': [Ridge()],
     'regressor__model__alpha': [0.1, 1, 10, 100]},
    {'regressor__model': [Lasso(max_iter=5000)],
     'regressor__model__alpha': [0.0001, 0.001, 0.01]}]
linear_search = models.tune_params({
    'Lineaire regressie': (linear_model(LINEAR_CAT, LINEAR_NUM), linear_grid,
                           LINEAR_CAT + LINEAR_NUM)})
models.make_submission('Lineaire regressie', 'submission_linear.csv')
show_grid(linear_search)

**Resultaat lineaire regressie.** CV-RMSE 59,3 (herfst 69,3, winter 45,9), testset-RMSE 85,6
($R^2$ = 0,75).

- Regularisatie geeft **geen verbetering**: gewone lineaire regressie en Ridge met $\alpha = 0{,}1$
  scoren gelijk (59,3). Met ±16.000 rijen en ±80 features is het risico op overfitting klein. Bij een
  grote $\alpha$ (Ridge $\alpha = 100$: 94,9; Lasso $\alpha = 0{,}01$: 120,0) worden de gewichten te
  sterk naar 0 gedrukt → **underfitting** (te veel bias).
- In de **winter** scoort het model veel beter dan in de herfst. Op log-schaal werken de effecten als
  percentages, en dat past bij de rustige winter. Dit klopt met de Kaggle-score van ronde 1 (56).
- Beperking: het effect van het weer is op elk uur even groot (geen interactie weer × uur).

## 3.4 Model 2: Beslisboom (regressieboom)

**Werking.** Een regressieboom verdeelt de data stap voor stap met ja/nee-vragen (bijvoorbeeld
`hour <= 6.5`). Bij elke splitsing kiest het CART-algoritme (Breiman et al., 1984) de feature $j$
en drempel $s$ die de som van de gekwadrateerde fouten in de twee nieuwe groepen minimaliseert:
$$\min_{j,\,s}\left[\sum_{x_i \in L}(y_i - \bar{y}_L)^2 + \sum_{x_i \in R}(y_i - \bar{y}_R)^2\right]$$
De voorspelling in een blad is het gemiddelde $\bar{y}$ van de trainingsrijen in dat blad.

**Keuzes.** Een boom vindt interacties (uur × werkdag × weer) zelf, dus we gebruiken de gewone
features zonder dummies, schaling of log-transformatie. Hyperparameters tegen overfitting:
`max_depth` en `min_samples_leaf`, net als de grid voor de beslisboom in opdracht 12 van
*Regression Time Series 2* (De Haagse Hogeschool, z.d.-d). **Let op:** een boom kan een trend niet
**extrapoleren**: voor waarden buiten de training krijgt hij de waarde van het meest rechtse blad
(*Week 10 application remarks*, De Haagse Hogeschool, z.d.-i; Kaggle-cursus les 5, Holbrook,
z.d.-b). Daarom geven we bomen `year` (niveau per jaar) in plaats van een doorlopende `trend`.

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 10 en 12

tree_grid = {'max_depth': [6, 10, 14, None], 'min_samples_leaf': [1, 5, 20]}
models.tune_params({
    'Beslisboom': (DecisionTreeRegressor(random_state=RANDOM_STATE),
                   tree_grid, TREE_FEATURES)})
models.make_submission('Beslisboom', 'submission_tree.csv')
models.show_feature_importances('Beslisboom')

## 3.5 Model 3: Random Forest (bagging-ensemble)

**Werking.** Random Forest (Breiman, 2001) combineert $B$ bomen:

1. **Bootstrap:** elke boom krijgt een steekproef met teruglegging uit de trainingsdata (*bagging*);
2. bij elke splitsing wordt een **willekeurige subset van de features** bekeken (`max_features`);
3. de voorspelling is het **gemiddelde**: $\hat{f}(x) = \frac{1}{B}\sum_{b=1}^{B} T_b(x)$.

**Waarom werkt dit?** Als elke boom variantie $\sigma^2$ heeft en de bomen onderling correlatie
$\rho$, dan is (Hastie et al., 2009)
$$\text{Var}\big(\hat{f}(x)\big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2$$
Meer bomen en minder gecorreleerde bomen geven dus een lagere variantie, terwijl de bias ongeveer
gelijk blijft.

We tunen het Random Forest met grid search en de tijdreeks-folds, zoals in opdracht 11 van
*Regression Time Series 2* (De Haagse Hogeschool, z.d.-d).

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 11 en 12

rf_grid = {'n_estimators': [200], 'max_features': [0.5, 1.0],
           'min_samples_leaf': [1, 3]}
models.tune_params({
    'Random Forest': (RandomForestRegressor(random_state=RANDOM_STATE),
                      rf_grid, TREE_FEATURES)})
models.make_submission('Random Forest', 'submission_rf.csv')
models.show_feature_importances('Random Forest')

## 3.6 Model 4: Gradient Boosting

**Werking.** Gradient boosting (Friedman, 2001; *Regression 1*, De Haagse Hogeschool, z.d.-b)
bouwt bomen **na elkaar**. Elke
nieuwe boom $f_t$ leert de **negatieve gradiënt** van de loss ten opzichte van de huidige voorspelling.
Voor de squared-error loss $l = \frac12(y - \hat{y})^2$ is dat precies het **residu** $y - \hat{y}$:
$$\hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta\, f_t(x), \qquad f_t \approx y - \hat{y}^{(t-1)}$$
De *learning rate* $\eta$ maakt elke stap kleiner, zodat het model geleidelijk leert (minder
overfitting). Met `subsample < 1` krijgt elke boom een deel van de rijen (*stochastic gradient
boosting*).

**Verschil met Random Forest:** bagging verlaagt vooral de **variantie** (onafhankelijke bomen
middelen), boosting verlaagt vooral de **bias** (bomen corrigeren elkaars fouten).

In [ ]:
# Bron: Week 8 – Regression 1, opdracht 1 (GradientBoostingRegressor)

gb_model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                     subsample=0.8,
                                     random_state=RANDOM_STATE)
models.tune_params({
    'Gradient Boosting': (gb_model, {'max_depth': [4, 6]}, TREE_FEATURES)})
models.make_submission('Gradient Boosting', 'submission_gb.csv')
models.show_feature_importances('Gradient Boosting')

## 3.7 Model 5: XGBoost (boosting-ensemble)

**Werking.** XGBoost (*Extreme Gradient Boosting*; Chen & Guestrin, 2016) is een snelle, uitgebreide
versie van gradient boosting **met regularisatie**. De objective is:
$$\mathcal{L} = \sum_{i=1}^{n} l(y_i, \hat{y}_i) + \sum_{k}\Omega(f_k), \qquad
\Omega(f) = \gamma T + \tfrac{1}{2}\lambda\sum_{j=1}^{T} w_j^2$$
met $l$ de loss function (squared error), $T$ het aantal bladeren en $w_j$ de waarde van blad $j$.
$\Omega$ straft grote en complexe bomen af.

XGBoost gebruikt de eerste en tweede afgeleide van de loss: $g_i = \partial l / \partial \hat{y}_i$
en $h_i = \partial^2 l / \partial \hat{y}_i^2$. Voor squared error is $g_i = \hat{y}_i - y_i$ en
$h_i = 1$. Het beste gewicht van blad $j$ is dan het **gemiddelde residu, gekrompen door $\lambda$**:
$$w_j^* = -\frac{G_j}{H_j + \lambda} = \frac{\sum_{i \in j}(y_i - \hat{y}_i)}{n_j + \lambda}$$
Een splitsing wordt gekozen op basis van de *gain* (en alleen gedaan als die positief is):
$$\text{Gain} = \frac12\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda}
- \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right] - \gamma$$

**Hyperparameters:** `n_estimators`, `learning_rate` ($\eta$), `max_depth` en `min_child_weight`
(minimale som van $h_i$ in een blad). We tunen ze met grid search, zoals in het hoofdstuk
*Fine-tuning your XGBoost model* van de DataCamp-cursus van week 8 (Fogelson, z.d.). Net als andere
bomen kan XGBoost (met `gbtree`) een trend niet extrapoleren; met `gblinear` zou dat wel kunnen
(*Week 10 application remarks*, De Haagse Hogeschool, z.d.-i).

In [ ]:
# Bron: Week 8 – DataCamp Extreme Gradient Boosting with XGBoost,
#       hoofdstuk Fine-tuning your XGBoost model (grid search)
# Bron: Kaggle Time Series, les 5 – Hybrid Models (Holbrook)

xgb_grid = {'max_depth': [4, 6, 8], 'n_estimators': [400, 800],
            'min_child_weight': [1, 5]}
xgb_model = XGBRegressor(learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, n_jobs=1,
                         random_state=RANDOM_STATE)
models.tune_params({'XGBoost': (xgb_model, xgb_grid, TREE_FEATURES)})
models.make_submission('XGBoost', 'submission_xgb.csv')
models.show_feature_importances('XGBoost')

**Resultaat boommodellen.**

| Model | CV-RMSE | herfst | winter | test-RMSE |
|---|---|---|---|---|
| Beslisboom | 70,3 | 75,5 | 63,3 | 104,8 |
| Random Forest | 55,0 | 58,9 | 49,8 | 90,4 |
| Gradient Boosting | 50,4 | 56,5 | 42,3 | 78,7 |
| XGBoost | 47,6 | 53,2 | 40,1 | 61,0 |

- **Eén beslisboom** (beste: geen maximale diepte, minimaal 5 rijen per blad) heeft een hoge variantie
  en is in de cross-validatie niet beter dan de baseline (beide 70,3).
- **Bagging** (Random Forest) verlaagt de variantie: 15 punten beter. **Boosting** verlaagt ook de
  bias en scoort het best: Gradient Boosting 50,4, XGBoost 47,6 (`max_depth` 6, `min_child_weight` 5,
  800 bomen).
- Op de testset (met twee feestdagen, zie 4.2) scoren alle boommodellen slechter dan in de
  cross-validatie. Het hybride model, dat de trend met lineaire regressie volgt, doet het daar veel
  beter (3.10).
- **Feature importances:** in de beslisboom, Random Forest en Gradient Boosting is `hour` veruit het
  belangrijkst (±0,6), gevolgd door `temp`, `year` en `workingday`. XGBoost (gemeten met *gain*) zet
  `workingday`, `hour` en `year` bovenaan. De importances verschillen per model en per meetmethode; in
  5.1 meten we de invloed eerlijker met permutation importance.

## 3.8 Tijdreeksmodel 1: SARIMAX

**Werking.** SARIMAX staat voor *Seasonal AutoRegressive Integrated Moving Average with eXogenous
variables* (Radhakrishnan, 2023d; Fulton, z.d.; Box et al., 2015):

- **AR($p$):** $y_t$ hangt af van eerdere waarden: $\phi_1 y_{t-1} + \dots + \phi_p y_{t-p}$.
- **I($d$):** het aantal keer differentiëren om de reeks stationair te maken (2.2).
- **MA($q$):** $y_t$ hangt af van eerdere voorspelfouten: $\theta_1\varepsilon_{t-1} + \dots +
  \theta_q\varepsilon_{t-q}$.
- **S – seizoensdeel $(P, D, Q)_s$:** hetzelfde met lags van een seizoen, bijv. $y_{t-24}$ bij
  $s = 24$.
- **X – exogene variabelen:** een regressie $y_t = \boldsymbol\beta^\top\mathbf{x}_t + \eta_t$
  waarbij de fout $\eta_t$ een (S)ARIMA-proces volgt.

In één formule, met de backshift-operator $B y_t = y_{t-1}$:
$$\phi_p(B)\,\Phi_P(B^s)\,(1 - B)^d(1 - B^s)^D\,(y_t - \boldsymbol\beta^\top\mathbf{x}_t)
= \theta_q(B)\,\Theta_Q(B^s)\,\varepsilon_t$$

We volgen het **Box-Jenkins-stappenplan** uit de les (Radhakrishnan, 2023d; Fulton, z.d.):
(1) identificeren, (2) schatten, (3) de residuen controleren en (4) voorspellen.

**1. Identificeren.** De uurreeks is stationair (ADF, 2.2) → $d = 0$. De PACF (2.5) heeft grote
waarden bij lag 1–2 en rond lag 24 → AR-termen en eventueel een seizoens-AR met $s = 24$. De keuze
uit de flowchart van de les (Radhakrishnan, 2023d): er is seizoen én er zijn externe variabelen
(weer) → **SARIMAX**.

**Keuzes.** Target $\log(1 + \text{cnt})$; exogene variabelen: 48 dummies voor uur × werkdag (het
dagpatroon), weersituatie, `temp`, `hum` en `windspeed`. SARIMAX heeft een regelmatige reeks nodig
(ontbrekende uren worden geïnterpoleerd). We trainen op de **laatste 56 dagen**: het verband met
bijvoorbeeld temperatuur verschilt per seizoen, recente data is het meest relevant en de rekentijd
blijft beperkt. Het model zelf (`SARIMAX`, `get_forecast`, `conf_int`) gebruiken we zoals in
opdracht 2.3 en 2.4 van *Time series analysis 3* (De Haagse Hogeschool, z.d.-h). De klasse
`SarimaxModel` eromheen (met `fit` en `predict` voor de grid search) komt niet in de lesstof voor
(Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook).

[ChatGPT, 2026. Prompt 12: SARIMAX in ons notebook](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)


In [ ]:
# Bron: Week 11 – Time series analysis 3, opdracht 2.3 en 2.4 (SARIMAX,
#       get_forecast en conf_int)
# Bron: Week 11 – DataCamp ARIMA Models in Python
# Prompt 1: Portfolio 2 CityBike notebook (klasse SarimaxModel)
# Prompt 12: SARIMAX in ons notebook

class SarimaxModel(BaseEstimator, RegressorMixin):
    """SARIMAX-model met dezelfde fit/predict-vorm als scikit-learn."""

    def __init__(self, order=(1, 0, 0), seasonal_order=(0, 0, 0, 0),
                 n_days=56):
        self.order = order
        self.seasonal_order = seasonal_order
        self.n_days = n_days

    @staticmethod
    def _regular(X, y=None):
        """Maak een regelmatige uurreeks; ontbrekende uren invullen."""
        data = X.copy()
        if y is not None:
            data['cnt'] = np.asarray(y)
        data = data.set_index('date_hour').asfreq('h')
        data[['holiday', 'weathersit']] = (data[['holiday', 'weathersit']]
                                           .ffill())
        data = data.interpolate().bfill()
        is_weekday = data.index.dayofweek < 5
        data['workingday'] = is_weekday & (data['holiday'] == 0)
        return data

    @staticmethod
    def _exog(data):
        """Exogene variabelen: uur x werkdag-dummies en het weer."""
        exog = pd.DataFrame(index=data.index)
        workingday = data['workingday'].astype(bool)
        for hour in range(24):
            is_hour = data.index.hour == hour
            exog[f'wd_{hour}'] = (is_hour & workingday).astype(float)
            exog[f'we_{hour}'] = (is_hour & ~workingday).astype(float)
        exog['weather_2'] = (data['weathersit'] == 2).astype(float)
        exog['weather_3'] = (data['weathersit'] >= 3).astype(float)
        for column in ['temp', 'hum', 'windspeed']:
            exog[column] = data[column]
        return exog

    def fit(self, X, y):
        data = self._regular(X, y)
        start = data.index.max() - pd.Timedelta(days=self.n_days)
        data = data[data.index > start]
        model = SARIMAX(np.log1p(data['cnt']), exog=self._exog(data),
                        order=self.order, seasonal_order=self.seasonal_order)
        self.result_ = model.fit(disp=False, maxiter=200)
        self.last_time_ = data.index.max()
        return self

    def _future(self, X):
        """Alle uren vanaf het einde van de training t/m X."""
        hours = pd.date_range(self.last_time_ + pd.Timedelta(hours=1),
                              X['date_hour'].max(), freq='h')
        future = self._regular(X).reindex(hours).ffill().bfill()
        return hours, future

    def predict_interval(self, X, alpha=0.05):
        """Voorspelling met betrouwbaarheidsinterval (get_forecast)."""
        hours, future = self._future(X)
        forecast = self.result_.get_forecast(len(hours),
                                             exog=self._exog(future))
        mean = pd.Series(np.expm1(forecast.predicted_mean.to_numpy()),
                         index=hours)
        interval = np.expm1(forecast.conf_int(alpha=alpha))
        interval.index = hours
        return mean, interval

    def predict(self, X):
        mean, _ = self.predict_interval(X)
        return mean.reindex(X['date_hour']).to_numpy()

**2. Schatten.** De orde is niet altijd goed af te lezen uit de plots; dan vergelijk je modellen met
**AIC** en **BIC** (lager is beter; ze belonen een goede fit en straffen extra parameters af;
Radhakrishnan, 2023d; Fulton, z.d.). We
fitten een paar kandidaten op de laatste 56 dagen van de trainset. Daarna kiezen we de orde ook met
de cross-validatie (de RMSE is wat Kaggle meet).

In [ ]:
# Bron: Week 11 – slides Timeseries analyse 3 (orde kiezen met AIC/BIC)
# Bron: Week 11 – DataCamp ARIMA Models in Python (AIC en BIC)

candidates = [((1, 0, 0), (0, 0, 0, 0)), ((2, 0, 0), (0, 0, 0, 0)),
              ((1, 0, 1), (0, 0, 0, 0)), ((2, 0, 1), (0, 0, 0, 0)),
              ((1, 0, 1), (1, 0, 0, 24))]
aic_rows = []
for order, seasonal_order in candidates:
    fitted = SarimaxModel(order, seasonal_order).fit(
        models.train[TS_FEATURES], models.train['cnt'])
    aic_rows.append({'order': order, 'seasonal_order': seasonal_order,
                     'AIC': fitted.result_.aic, 'BIC': fitted.result_.bic})
display(pd.DataFrame(aic_rows).round(0))

sarimax_grid = [
    {'order': [(1, 0, 0), (2, 0, 0), (1, 0, 1)],
     'seasonal_order': [(0, 0, 0, 0)]},
    {'order': [(1, 0, 1)], 'seasonal_order': [(1, 0, 0, 24)]}]
sarimax_search = models.tune_params({
    'SARIMAX': (SarimaxModel(), sarimax_grid, TS_FEATURES)})
models.make_submission('SARIMAX', 'submission_sarimax.csv')
show_grid(sarimax_search)

**3. Residuen controleren.** Een goed model laat alleen **witte ruis** over: residuen zonder patroon,
die niet van vorige waarden afhangen (Radhakrishnan, 2023d). `plot_diagnostics` toont de residuen,
hun verdeling (histogram en QQ-plot) en hun autocorrelatie (correlogram), zoals in de DataCamp-cursus
*ARIMA Models in Python* van week 11 (Fulton, z.d.).

In [ ]:
# Bron: Week 11 – DataCamp ARIMA Models in Python (plot_diagnostics)
# Bron: Week 11 – slides Timeseries analyse 3 (Box-Jenkins: witte ruis)

best_sarimax = models.models['SARIMAX'][0]
best_sarimax.result_.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

**4. Voorspellen.** Met `get_forecast` krijgen we de voorspelling én een **95%-betrouwbaarheids-
interval**, zoals in opdracht 2.4 van *Time series analysis 3* (De Haagse Hogeschool, z.d.-h). Het
interval laat de **voortplanting van fouten** zien: hoe verder vooruit, hoe onzekerder de
voorspelling (Radhakrishnan, 2023d).

In [ ]:
# Bron: Week 11 – Time series analysis 3, opdracht 2.4 (interval plotten)

mean, interval = best_sarimax.predict_interval(models.test[TS_FEATURES])
actual = models.test.set_index('date_hour')['cnt']
first_week = mean.index[:7 * 24]

plt.figure(figsize=(14, 5))
plt.plot(actual.loc[first_week[0]:first_week[-1]], label='werkelijk')
plt.plot(first_week, mean.loc[first_week], color='red', label='SARIMAX')
plt.fill_between(first_week, interval.iloc[:7 * 24, 0],
                 interval.iloc[:7 * 24, 1], color='pink', alpha=0.5,
                 label='95%-interval')
plt.title('SARIMAX-voorspelling met 95%-betrouwbaarheidsinterval '
          '(eerste week van de testset)')
plt.legend()
plt.show()
# Breedte van het interval op log-schaal, per aantal uren vooruit
log_width = np.log1p(interval.iloc[:, 1]) - np.log1p(interval.iloc[:, 0])
steps = [1, 2, 3, 6, 24, len(log_width)]
pd.Series(log_width.to_numpy()[[step - 1 for step in steps]],
          index=[f'{step} uur vooruit' for step in steps],
          name='breedte 95%-interval (log-schaal)').round(2)

**Resultaat SARIMAX.**

- **AIC/BIC:** het model met seizoens-AR, (1,0,1)(1,0,0,24), past statistisch het best (laagste AIC
  408 en BIC 705). Voor voorspellingen 19 dagen vooruit maakt het echter geen verschil: CV-RMSE 63,9
  tegen **63,8** voor het eenvoudige AR(1)-model. We kiezen het eenvoudigste en snelste model:
  **AR(1) met exogene variabelen**.
- **Residuen:** rond 0 en bijna normaal verdeeld, met dikke staarten (QQ-plot: uren met heel weinig
  verhuur en feestdagen). Het correlogram toont bijna geen autocorrelatie meer: de residuen zijn
  vrijwel **witte ruis**, dus het model heeft de structuur gevangen.
- **Interval:** de breedte (op log-schaal) groeit van 1,06 (1 uur vooruit) naar 1,94 (6 uur) en
  2,14 (24 uur), en blijft daarna gelijk tot 456 uur. Dat is de **voortplanting van fouten**: na
  ongeveer een dag is het AR-deel uitgedoofd en hangt de voorspelling alleen nog van de exogene
  variabelen af. Omgerekend naar fietsen is het interval in de spits erg breed.
- **Scores:** CV-RMSE 63,8 (winter 50,1), testset 90,8. SARIMAX gebruikt maar 8 weken data en leert
  geen interacties tussen weer en uur. Op Kaggle (ronde 1) scoorde het relatief goed (55).

## 3.9 Tijdreeksmodel 2: Prophet

**Motivatie.** Als tweede tijdreeksmodel kiezen we Prophet (Taylor & Letham, 2018). Het bouwt
direct voort op de stof: een **trend** (Kaggle-cursus les 2, Holbrook, z.d.-f) en **seizoenen als
Fourier-reeksen** (Kaggle-cursus les 3, Holbrook, z.d.-d; Radhakrishnan, 2023c). Daarnaast kan het
met ontbrekende uren omgaan en exogene variabelen (weer) gebruiken. Prophet zelf zit niet in de
lesstof: de klasse `ProphetModel` is gemaakt met hulp van GenAI (Claude, 2026. Prompt 1:
Portfolio 2 CityBike notebook) en gecontroleerd met Taylor en Letham (2018).

**Werking.** Prophet is een additief model:
$$y(t) = g(t) + s(t) + h(t) + \boldsymbol\beta^\top\mathbf{x}(t) + \varepsilon_t$$

- $g(t)$: **trend**, stuksgewijs lineair met *changepoints* (momenten waarop de helling mag
  veranderen). `changepoint_prior_scale` bepaalt hoe flexibel de trend is (regularisatie).
- $s(t)$: **seizoenen** als Fourier-reeks met periode $P$:
  $s(t) = \sum_{n=1}^{N}\left(a_n\cos\frac{2\pi n t}{P} + b_n\sin\frac{2\pi n t}{P}\right)$, voor
  week en jaar. Omdat het dagpatroon op werkdagen en in het weekend verschilt (EDA), maken we **twee
  dagseizoenen** (*conditional seasonality*).
- $h(t)$ en $\mathbf{x}(t)$: feestdagen en het weer als extra regressors.

We modelleren $\log(1 + \text{cnt})$, zodat trend, seizoen en weer in de echte schaal met elkaar
vermenigvuldigen.

[ChatGPT, 2026. Prompt 13: Prophet in ons notebook](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)


In [ ]:
# Prompt 1: Portfolio 2 CityBike notebook (klasse ProphetModel)
# Zie ook (niet-GenAI): Taylor en Letham (2018)
# Prompt 13: Prophet in ons notebook


class ProphetModel(BaseEstimator, RegressorMixin):
    """Prophet-model met dezelfde fit/predict-vorm als scikit-learn."""

    REGRESSORS = ['holiday', 'temp', 'hum', 'windspeed', 'weather_2',
                  'weather_3']

    def __init__(self, changepoint_prior_scale=0.05):
        self.changepoint_prior_scale = changepoint_prior_scale

    @staticmethod
    def _to_prophet(X):
        """Zet de data om naar het formaat van Prophet (kolom 'ds')."""
        data = pd.DataFrame({'ds': X['date_hour'].to_numpy()})
        is_weekday = X['date_hour'].dt.dayofweek.to_numpy() < 5
        data['workingday'] = is_weekday & (X['holiday'].to_numpy() == 0)
        data['weekend'] = ~data['workingday']
        data['weather_2'] = (X['weathersit'].to_numpy() == 2).astype(int)
        data['weather_3'] = (X['weathersit'].to_numpy() >= 3).astype(int)
        for column in ['holiday', 'temp', 'hum', 'windspeed']:
            data[column] = X[column].to_numpy()
        return data

    def fit(self, X, y):
        logging.disable(logging.WARNING)       # logberichten van Prophet uit
        data = self._to_prophet(X)
        data['y'] = np.log1p(np.asarray(y))
        model = Prophet(daily_seasonality=False, weekly_seasonality=True,
                        yearly_seasonality=True,
                        changepoint_prior_scale=self.changepoint_prior_scale)
        model.add_seasonality('dag_werkdag', period=1, fourier_order=10,
                              condition_name='workingday')
        model.add_seasonality('dag_weekend', period=1, fourier_order=10,
                              condition_name='weekend')
        for column in self.REGRESSORS:
            model.add_regressor(column)
        self.model_ = model.fit(data)
        logging.disable(logging.NOTSET)        # logberichten weer aan
        return self

    def predict(self, X):
        forecast = self.model_.predict(self._to_prophet(X))
        return np.expm1(forecast['yhat'].to_numpy())


# n_jobs=1: Prophet start zelf al een apart proces (Stan)
prophet_search = models.tune_params(
    {'Prophet': (ProphetModel(), {'changepoint_prior_scale': [0.01, 0.05]},
                 TS_FEATURES)}, n_jobs=1)
models.make_submission('Prophet', 'submission_prophet.csv')
show_grid(prophet_search)

**Resultaat Prophet.** CV-RMSE 58,6 met `changepoint_prior_scale` = 0,01 (bij 0,05: 64,7). Een
**stijve trend** werkt het best: een flexibele trend volgt toevallige schommelingen en wordt daarna
verkeerd doorgetrokken (overfitting). Net als de andere modellen op log-schaal scoort Prophet in de
winter (45,0) veel beter dan in de herfst (68,9). Op de testset: 89,2.

## 3.10 Hybride model (BoostedHybrid)

**Werking.** Uit Kaggle-cursus les 5, *Hybrid Models* (Holbrook, z.d.-b), en de *Week 10
application remarks* (De Haagse Hogeschool, z.d.-i): bomen kunnen een trend niet extrapoleren, en
lineaire regressie mist complexe interacties. Een **hybride model** traint daarom twee modellen na
elkaar (in de les heet dit een *boosted hybrid*):

1. **Model 1 (lineaire regressie)** leert de trend en het jaarseizoen met de features uit
   `DeterministicProcess` (`trend` + Fourier-termen): $\hat{y}^{(1)}$. Dit kan extrapoleren.
2. **Model 2 (XGBoost)** leert wat model 1 niet kan verklaren (uur, werkdag, weer).
3. De voorspelling combineert beide.

We werken op **log-schaal**, zodat de trend een groeipercentage is en de correcties van model 2
percentages zijn. We testen twee versies van model 2:

- **kwadratische loss** (zoals in Kaggle-cursus les 5): XGBoost leert de residuen
  $r_i = \log(1 + y_i) - \hat{y}^{(1)}_i$, en $\hat{y} = e^{\hat{y}^{(1)} + \hat{r}} - 1$;
- **Poisson-loss** (onze verbetering, iteratie 2): zie hieronder. Deze variant zit niet in de
  lesstof en is gemaakt met hulp van GenAI (Claude, 2026. Prompt 1: Portfolio 2 CityBike
  notebook), gecontroleerd met McCullagh en Nelder (1989) en XGBoost Developers (z.d.).

**Waarom Poisson-loss?** Na een log-transformatie geeft terugrekenen met $e^{\hat{y}} - 1$
ongeveer de **mediaan** in plaats van het **gemiddelde**. Bij een scheve verdeling ligt de mediaan
lager, dus het model voorspelt systematisch te laag, terwijl de RMSE juist het gemiddelde beloont.
`cnt` is een **telling**. Een natuurlijk model voor tellingen is de Poisson-verdeling met gemiddelde
$\mu$ en $\log \mu = f(x)$ (*log-link*, zoals bij een GLM; McCullagh & Nelder, 1989). De loss function
is de negatieve log-likelihood:
$$l(y, \mu) = \mu - y \log \mu$$
Die is minimaal als $\mu$ het **gemiddelde** van $y$ is. Het model rekent dus met percentages én
voorspelt het gemiddelde. In XGBoost: `objective='count:poisson'`, met gradiënt $g_i = \mu_i - y_i$ en
hessiaan $h_i = \mu_i$ (XGBoost Developers, z.d.). Model 2 start bij de voorspelling van model 1 (de
*base margin*) en leert alleen de correcties:
$$\log \mu(x) = \hat{y}^{(1)}(x) + \sum_{t} \eta\, f_t(x), \qquad \hat{y} = \mu(x)$$
`max_delta_step = 0.7` houdt het leren stabiel (aanbevolen voor Poisson; XGBoost Developers, z.d.).

[ChatGPT, 2026. Prompt 14: Poisson-loss en base margin](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)


In [ ]:
# Bron: Kaggle Time Series, les 5 – Hybrid Models (Holbrook)
# Bron: Week 10 – application remarks (hybride model)
# Prompt 1: Portfolio 2 CityBike notebook (Poisson-loss met base_margin)
# Zie ook (niet-GenAI): XGBoost Developers (z.d.); McCullagh en Nelder
#       (1989)
# Prompt 14: Poisson-loss en base margin


class BoostedHybrid(BaseEstimator, RegressorMixin):
    """Hybride model (naar Holbrook, z.d.-b).

    Model 1: lineaire regressie op trend + Fourier-termen (log-schaal).
    Model 2: XGBoost op wat model 1 niet verklaart, met kwadratische loss
    (residuen) of Poisson-loss (base margin = voorspelling van model 1).
    """

    def __init__(self, trend_features=None, residual_features=None,
                 loss='squared', n_estimators=600, max_depth=6,
                 learning_rate=0.05, subsample=0.8, reg_lambda=1.0):
        self.trend_features = trend_features
        self.residual_features = residual_features
        self.loss = loss
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.reg_lambda = reg_lambda

    def _make_model_2(self):
        settings = dict(n_estimators=self.n_estimators,
                        max_depth=self.max_depth,
                        learning_rate=self.learning_rate,
                        subsample=self.subsample, reg_lambda=self.reg_lambda,
                        min_child_weight=5, colsample_bytree=0.8, n_jobs=1,
                        random_state=RANDOM_STATE)
        if self.loss == 'poisson':
            return XGBRegressor(objective='count:poisson',
                                max_delta_step=0.7, **settings)
        return XGBRegressor(objective='reg:squarederror', **settings)

    def fit(self, X, y):
        y = np.asarray(y)
        # Stap 1: trend en jaarseizoen op log-schaal
        self.model_1_ = LinearRegression()
        self.model_1_.fit(X[self.trend_features], np.log1p(y))
        base = self.model_1_.predict(X[self.trend_features])
        # Stap 2: model 2 leert wat model 1 niet verklaart
        self.model_2_ = self._make_model_2()
        if self.loss == 'poisson':
            self.model_2_.fit(X[self.residual_features], y, base_margin=base)
        else:
            self.model_2_.fit(X[self.residual_features], np.log1p(y) - base)
        return self

    def predict(self, X):
        base = self.model_1_.predict(X[self.trend_features])
        if self.loss == 'poisson':
            return self.model_2_.predict(X[self.residual_features],
                                         base_margin=base)
        residual = self.model_2_.predict(X[self.residual_features])
        return np.expm1(base + residual)


hybrid = BoostedHybrid(trend_features=TREND_FEATURES,
                       residual_features=TREE_FEATURES)
hybrid_features = TREND_FEATURES + TREE_FEATURES
squared_grid = {'loss': ['squared'], 'max_depth': [6, 8],
                'n_estimators': [300, 600]}
poisson_grid = {'loss': ['poisson'], 'n_estimators': [1000],
                'max_depth': [5, 6], 'subsample': [0.6, 0.8],
                'reg_lambda': [1, 10]}
models.tune_params({
    'Hybride model (kwadratisch)': (hybrid, squared_grid, hybrid_features)})
models.make_submission('Hybride model (kwadratisch)',
                       'submission_hybrid.csv')
poisson_search = models.tune_params({
    'Hybride model (Poisson)': (hybrid, poisson_grid, hybrid_features)})
models.make_submission('Hybride model (Poisson)',
                       'submission_hybrid_poisson.csv')
show_grid(poisson_search)

**Resultaat hybride modellen.**

- **Kwadratische loss** (zoals in Kaggle-cursus les 5): CV-RMSE **45,3** (winter 35,3), testset 50,4. Dat is al
  beter dan alle losse modellen: de lineaire trend van model 1 en de interacties van XGBoost vullen
  elkaar aan.
- **Poisson-loss:** CV-RMSE **42,8** (herfst 50,9, winter **32,0**), testset 50,8 ($R^2$ = 0,91).
  Beste parameters: `max_depth` = 5, 1000 bomen, `reg_lambda` = 1, `subsample` = 0,8. Ondiepere
  bomen (5) werken beter dan diepere (6): het model generaliseert beter.
- Op de testset zijn beide hybride modellen even goed (50,4 en 50,8). In de cross-validatie, en vooral
  in de **winter-folds** (het seizoen van de Kaggle-test), is de Poisson-versie duidelijk beter.
  Daarom kiezen we het **hybride model met Poisson-loss** als eindmodel.

# 4. Bevindingen

## 4.1 Overzicht van de resultaten

Per model: de gemiddelde RMSE over de 7 folds (CV-RMSE, apart voor herfst en winter), de scores op de
testset (12–30 november 2012) en de verbetering ten opzichte van de baseline (op de test-RMSE;
Radhakrishnan, 2025). De code voor de overzichtstabel en de grafiek komt niet in de lesstof voor
(Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook).

[ChatGPT, 2026. Prompt 15: Overzicht van de modelresultaten](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)

In [ ]:
# Bron: Week 8 – slides Veelvoorkomende fouten (vergelijk met baseline)
# Prompt 1: Portfolio 2 CityBike notebook (overzichtstabel)
# Prompt 15: Overzicht van de modelresultaten


results = models.results_table()
overview = results[['CV-RMSE', 'herfst', 'winter', 'test R2', 'test MAE',
                    'test RMSE']].astype(float).round(2)
improvement = results['verbetering t.o.v. baseline'].astype(float) * 100
overview['verbetering t.o.v. baseline'] = improvement.round(0).map(
    '{:.0f}%'.format)
overview['beste parameters'] = results['beste parameters']
BEST_MODEL = overview.index[0]
print('Beste model (laagste CV-RMSE):', BEST_MODEL)
overview

[ChatGPT, 2026. Prompt 16: RMSE-grafiek per model](https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14)

In [ ]:
# Prompt 1: Portfolio 2 CityBike notebook (grafiek RMSE per model)
# Prompt 16: RMSE-grafiek per model


ax = overview[['CV-RMSE', 'test RMSE']].plot(kind='barh', figsize=(10, 6))
ax.axvline(MIN_KAGGLE_RMSE, color='red', linestyle='--',
           label=f'eis Kaggle ({MIN_KAGGLE_RMSE})')
ax.invert_yaxis()
ax.set(title='RMSE per model (lager = beter)', xlabel='RMSE')
ax.legend()
plt.show()

## 4.2 Voorspellingen op de testset en voor december

We vergelijken het beste model met SARIMAX en de baseline in de eerste week van de testset. Daarna
tonen we de voorspelling van het beste model voor de Kaggle-periode (1–19 december 2012). Net als in
opdracht 2.4 van *Time series analysis 3* (De Haagse Hogeschool, z.d.-h) en de forecast-grafieken
van Kaggle-cursus les 2 (Holbrook, z.d.-f) tekenen we werkelijkheid en voorspelling in één figuur.

In [ ]:
# Bron: Week 11 – Time series analysis 3, opdracht 2.4 (voorspelling
#       en werkelijkheid in één grafiek)
# Bron: Kaggle Time Series, les 2 – Trend (Holbrook)

actual = models.test.set_index('date_hour')['cnt']
week = slice(0, 7 * 24)
plt.figure(figsize=(14, 5))
plt.plot(actual.index[week], actual.to_numpy()[week], color='black',
         label='werkelijk')
for name in [BEST_MODEL, 'SARIMAX', 'Baseline']:
    plt.plot(actual.index[week], models.test_predictions[name][week],
             label=name)
plt.title('Werkelijk vs voorspeld (eerste week van de testset)')
plt.ylabel('cnt')
plt.legend()
plt.show()

best_file = os.path.join('submissions', results.loc[BEST_MODEL, 'submission'])
december = pd.read_csv(best_file, parse_dates=['date_hour'])
recent = train.set_index('date_hour')['cnt'].loc['2012-11-10':]
plt.figure(figsize=(14, 4))
plt.plot(recent, label='trainingsdata (werkelijk)')
plt.plot(december['date_hour'], december['cnt'],
         label=f'voorspelling Kaggle-test ({BEST_MODEL})')
plt.title('Voorspelling voor 1 t/m 19 december 2012')
plt.ylabel('cnt')
plt.legend()
plt.show()

**Beknopt overzicht.**

- **Beste model: het hybride model met Poisson-loss.** CV-RMSE **42,8** (winter 32,0); op de testset
  RMSE **50,8**, MAE 30,9 en $R^2$ = 0,91.
- **Ten opzichte van de baseline** (gemiddelde per uur en dagtype van de laatste 4 weken; test-RMSE
  126) is de fout **60% lager**. Per uur zit het model gemiddeld ±31 fietsen naast de werkelijkheid
  (MAE), de baseline ±81.
- **Volgorde (CV):** hybride (Poisson) > hybride (kwadratisch) > XGBoost > Gradient Boosting > Random
  Forest > Prophet > lineaire regressie > SARIMAX > beslisboom ≈ baseline. Boosting met een lineaire
  trend scoort het best.
- Alle modellen halen in de cross-validatie een RMSE onder de eis van 80. Op de testset scoren
  meerdere modellen boven 80: in 12–30 november vallen **twee feestdagen** (12 en 22 november), met
  daarna een lang weekend met weinig verhuur. Op 12 november voorspellen alle modellen een
  weekendpatroon, terwijl er wél spitsen zijn (grafiek): niet iedereen is dan vrij. In de
  Kaggle-periode (1–19 december) vallen geen feestdagen; daar verwachten we lagere fouten (in ronde 1
  scoorden alle modellen tussen 47 en 74).
- De voorspelling voor december heeft een logisch patroon (spitsen op werkdagen, middagpiek in het
  weekend) en sluit aan op het niveau van eind november.

## 4.3 Scores op Kaggle

We hebben in meerdere rondes ingeleverd; zo is de verbetering zichtbaar.

**Ronde 1 (12-11-2026): eerste versie** — alleen herfst-folds, zonder weer van het vorige/volgende
uur.

| # | Model | Bestand | Kaggle public |
|---|---|---|---|
| 1 | XGBoost | submission_xgb.csv | 49,28 |
| 2 | Beslisboom | submission_tree.csv | 62,11 |
| 3 | SVR | submission_svr.csv | 63,41 |
| 4 | SARIMAX | submission_sarimax.csv | 54,85 |
| 5 | Random Forest | submission_rf.csv | 66,14 |
| 6 | Prophet | submission_prophet.csv | 73,83 |
| 7 | Lineaire regressie | submission_linear.csv | 56,15 |
| 8 | Hybride model | submission_hybrid.csv | 49,43 |
| 9 | Gemiddelde XGBoost + hybride | submission_average.csv | **47,45** |

**Wat leerden we hiervan?** Alle modellen haalden de eis (< 80). Maar lineaire regressie (56) en
SARIMAX (55) scoorden op Kaggle veel beter dan in onze herfst-validatie, en relatief beter dan Random
Forest en SVR. Beide werken op log-schaal (percentages), en SARIMAX traint alleen op recente weken.
Conclusie: de herfst-folds waren niet representatief voor december. Daarom hebben we (1) winter-folds
toegevoegd (2.1), (2) het weer van het vorige/volgende uur toegevoegd (2.6) en (3) het hybride
model verbeterd met Poisson-loss (3.10).

**Ronde 2 en 3: verbeterde versies (dit notebook)**

> **Invullen na het uploaden op Kaggle.**

| # | Datum | Model | Bestand | CV-RMSE | test-RMSE | Kaggle public |
|---|---|---|---|---|---|---|
| 10 | | Hybride model (Poisson), versie 2 | submission_poisson_hybrid.csv | | | |
| 11 | | Hybride model (Poisson), dit notebook | submission_hybrid_poisson.csv | 42,8 | 50,8 | |
| 12 | | *(andere modellen uit dit notebook)* | | | | |

*Screenshot van de Kaggle-submissions invoegen.*

## Bronnen bij hoofdstuk 3 en 4

Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time series analysis:
Forecasting and control* (5th ed.). Wiley.

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.
https://doi.org/10.1023/A:1010933404324

Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and regression
trees*. Wadsworth.

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. In *Proceedings of the
22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 785–794).
ACM. https://doi.org/10.1145/2939672.2939785

Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook. Taalmodel Claude Opus 5.5 (Anthropic),
gesprek van november 2026. *[link naar het gedeelde gesprek invullen]*

De Haagse Hogeschool. (z.d.-b). *Regression 1* [Jupyter-notebook, 26/27 Machine Learning, week 8].
Brightspace.

De Haagse Hogeschool. (z.d.-d). *Regression Time Series 2* [Jupyter-notebook, 26/27 Machine
Learning, week 10 en 11]. Brightspace.

De Haagse Hogeschool. (z.d.-e). *Regression Time Series 3* [Jupyter-notebook, 26/27 Machine
Learning, week 12]. Brightspace.

De Haagse Hogeschool. (z.d.-h). *Time series analysis 3: Machine learning with timeseries*
[Jupyter-notebook, 26/27 Machine Learning, week 11 en 12]. Brightspace.

De Haagse Hogeschool. (z.d.-i). *Week 10 application remarks* [PowerPoint-slides, 26/27 Machine
Learning, week 10]. Brightspace.

Fogelson, S. (z.d.). *Extreme gradient boosting with XGBoost* [Online cursus, week 8]. DataCamp.
https://www.datacamp.com/courses/extreme-gradient-boosting-with-xgboost

Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of
Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

Fulton, J. (z.d.). *ARIMA models in Python* [Online cursus, week 11]. DataCamp.
https://www.datacamp.com/courses/arima-models-in-python

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The elements of statistical learning: Data
mining, inference, and prediction* (2nd ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal
problems. *Technometrics, 12*(1), 55–67. https://doi.org/10.1080/00401706.1970.10488634

Holbrook, R. (z.d.-b). *Hybrid models* [Kaggle-notebook; Kaggle-cursus Time Series, les 5]. Kaggle.
Geraadpleegd op 16 november 2026, van https://www.kaggle.com/code/ryanholbrook/hybrid-models

Holbrook, R. (z.d.-d). *Seasonality* [Kaggle-notebook; Kaggle-cursus Time Series, les 3]. Kaggle.
Geraadpleegd op 16 november 2026, van https://www.kaggle.com/code/ryanholbrook/seasonality

Holbrook, R. (z.d.-f). *Trend* [Kaggle-notebook; Kaggle-cursus Time Series, les 2]. Kaggle.
Geraadpleegd op 16 november 2026, van https://www.kaggle.com/code/ryanholbrook/trend

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical
learning: With applications in R* (2nd ed.). Springer. https://doi.org/10.1007/978-1-0716-1418-1

McCullagh, P., & Nelder, J. A. (1989). *Generalized linear models* (2nd ed.). Chapman & Hall.

Radhakrishnan, V. M. (2023a). *Parameters & hyperparameters* [PowerPoint-slides, week 8]. De Haagse
Hogeschool.

Radhakrishnan, V. M. (2023c). *Timeseries analyse 2 – Frequency domain (spectral analysis)*
[PowerPoint-slides, week 10]. De Haagse Hogeschool.

Radhakrishnan, V. M. (2023d). *Timeseries analyse 3 – Timeseries & machine learning*
[PowerPoint-slides, week 11]. De Haagse Hogeschool.

Radhakrishnan, V. M. (2025). *Veelvoorkomende fouten* [PowerPoint-slides, week 8]. De Haagse
Hogeschool.

Taylor, S. J., & Letham, B. (2018). Forecasting at scale. *The American Statistician, 72*(1),
37–45. https://doi.org/10.1080/00031305.2017.1380080

Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal
Statistical Society: Series B (Methodological), 58*(1), 267–288.
https://doi.org/10.1111/j.2517-6161.1996.tb02080.x

XGBoost Developers. (z.d.). *XGBoost parameters*. XGBoost documentation. Geraadpleegd op 12 november
2026, van https://xgboost.readthedocs.io/en/stable/parameter.html

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 10: Resultatentabel en grid search, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 11: Seizoensbaseline, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 12: SARIMAX in ons notebook, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 13: Prophet in ons notebook, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 14: Poisson-loss en base margin, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 15: Overzicht van de modelresultaten, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14

OpenAI. (2026, 19 november). *Uitleg resultaten en grid search* [ChatGPT, GPT-6 met Medium effort, prompt 16: RMSE-grafiek per model, uitleg van bestaande code]. https://chatgpt.com/share/6aff017d-2684-83ed-8c5d-5113764cba14
